# Modélisation — Prédiction du résultat d'un match

Objectif : comparer trois modèles sur une séparation chronologique. Les matchs les plus récents constituent le test, ce qui reproduit une vraie prédiction future.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss, mean_absolute_error, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier


In [ ]:
data = pd.read_csv("../data/processed/modeling_data.csv", parse_dates=["date"])
data = data.sort_values(["date", "match_id"]).reset_index(drop=True)
print("Dimensions :", data.shape)
print("Période :", data["date"].min(), "à", data["date"].max())
display(data["target"].value_counts(normalize=True).round(3))


## Sélection des variables

Les scores, la cible, les dates de classement et les identifiants sont exclus. Les noms d'équipes ne sont pas utilisés : le modèle doit apprendre la force et la forme, pas mémoriser une sélection.

In [ ]:
numeric_features = [
    "home_recent_matches_count", "home_recent_wins", "home_recent_draws", "home_recent_losses",
    "home_avg_goals_for", "home_avg_goals_against",
    "away_recent_matches_count", "away_recent_wins", "away_recent_draws", "away_recent_losses",
    "away_avg_goals_for", "away_avg_goals_against",
    "recent_wins_diff", "recent_draws_diff", "recent_losses_diff",
    "avg_goals_for_diff", "avg_goals_against_diff",
    "home_recent_form_points", "away_recent_form_points", "recent_form_points_diff",
    "home_rank", "away_rank", "home_total_points", "away_total_points",
    "home_rank_change", "away_rank_change", "rank_diff", "total_points_diff",
    "home_ranking_age_days", "away_ranking_age_days",
]
categorical_features = ["tournament", "home_confederation", "away_confederation", "neutral"]
feature_columns = numeric_features + categorical_features

rows_before = len(data)
data = data.dropna(subset=feature_columns).reset_index(drop=True)
print(f"Lignes retirées pour valeurs manquantes (classement FIFA indisponible) : {rows_before - len(data)} ({(rows_before - len(data)) / rows_before:.1%})")

onehot_features = ["tournament", "home_confederation", "away_confederation"]
X = pd.get_dummies(data[feature_columns], columns=onehot_features)
X["neutral"] = X["neutral"].astype(int)
encoded_columns = X.columns.tolist()
y = data["target"]

cutoff_date = data["date"].quantile(0.80)
train_mask = data["date"] < cutoff_date
test_mask = ~train_mask
X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]
assert data.loc[train_mask, "date"].max() < data.loc[test_mask, "date"].min()
print("Train :", len(X_train), "— jusqu'au", data.loc[train_mask, "date"].max())
print("Test :", len(X_test), "— à partir du", data.loc[test_mask, "date"].min())


In [ ]:
def scale_numeric(df, mean, std):
    scaled = df.copy()
    scaled[numeric_features] = (df[numeric_features] - mean) / std
    return scaled

train_mean = X_train[numeric_features].mean()
train_std = X_train[numeric_features].std().replace(0, 1)
X_train_scaled = scale_numeric(X_train, train_mean, train_std)
X_test_scaled = scale_numeric(X_test, train_mean, train_std)

validation_cutoff = data.loc[train_mask, "date"].quantile(0.80)
subtrain_mask = train_mask & (data["date"] < validation_cutoff)
validation_mask = train_mask & ~subtrain_mask
depth_results = []
for depth in [3, 5, 8, 12, None]:
    candidate = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=10, class_weight="balanced", random_state=42)
    candidate.fit(X.loc[subtrain_mask], y.loc[subtrain_mask])
    depth_results.append({"max_depth": depth, "validation_macro_f1": f1_score(y.loc[validation_mask], candidate.predict(X.loc[validation_mask]), average="macro")})
depth_results_df = pd.DataFrame(depth_results).sort_values("validation_macro_f1", ascending=False).reset_index(drop=True)
display(depth_results_df)
best_depth = depth_results_df.loc[0, "max_depth"]
best_depth = None if pd.isna(best_depth) else int(best_depth)

models = {
    "Logistic Regression": LogisticRegression(C=0.5, max_iter=600, tol=1e-3, class_weight={0: 1, 1: 1.6, 2: 1}, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=best_depth, min_samples_leaf=10, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=42),
}


In [ ]:
results, fitted_models = [], {}
for name, model in models.items():
    fit_X_train, fit_X_test = (X_train_scaled, X_test_scaled) if name == "Logistic Regression" else (X_train, X_test)
    model.fit(fit_X_train, y_train)
    train_predictions = model.predict(fit_X_train)
    predictions = model.predict(fit_X_test)
    probabilities = model.predict_proba(fit_X_test)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision_weighted": precision_score(y_test, predictions, average="weighted"),
        "recall_weighted": recall_score(y_test, predictions, average="weighted"),
        "f1_weighted": f1_score(y_test, predictions, average="weighted"),
        "train_macro_f1": f1_score(y_train, train_predictions, average="macro"),
        "macro_f1": f1_score(y_test, predictions, average="macro"),
        "draw_recall": recall_score(y_test, predictions, labels=[1], average=None, zero_division=0)[0],
        "log_loss": log_loss(y_test, probabilities, labels=model.classes_),
    })
    fitted_models[name] = model

results_df = pd.DataFrame(results).sort_values(["macro_f1", "log_loss"], ascending=[False, True]).reset_index(drop=True)
display(results_df.round(4))
best_name = results_df.loc[0, "model"]
best_test_model = fitted_models[best_name]
best_test_X = X_test_scaled if best_name == "Logistic Regression" else X_test
best_predictions = best_test_model.predict(best_test_X)
print("Meilleur modèle :", best_name)
print(classification_report(y_test, best_predictions))
display(pd.DataFrame(confusion_matrix(y_test, best_predictions, labels=best_test_model.classes_), index=best_test_model.classes_, columns=best_test_model.classes_))


## Réentraînement et sauvegarde pour Django

Après l'évaluation honnête sur le test, le meilleur type de modèle est réentraîné sur tout l'historique disponible. Le bundle contient aussi les features et les classes nécessaires à l'application.

In [ ]:
final_mean = X[numeric_features].mean()
final_std = X[numeric_features].std().replace(0, 1)
X_scaled_full = scale_numeric(X, final_mean, final_std)

final_model = models[best_name]
if best_name == "Logistic Regression":
    final_model.fit(X_scaled_full, y)
else:
    final_model.fit(X, y)

bundle = {
    "model": final_model,
    "feature_columns": feature_columns,
    "onehot_features": onehot_features,
    "encoded_columns": encoded_columns,
    "numeric_features": numeric_features,
    "scaler_mean": final_mean.to_dict() if best_name == "Logistic Regression" else None,
    "scaler_std": final_std.to_dict() if best_name == "Logistic Regression" else None,
    "classes": final_model.classes_.tolist(), "class_mapping": {0: "home_win", 1: "draw", 2: "away_win"}, "model_name": best_name,
    "training_end_date": data["date"].max().strftime("%Y-%m-%d"),
    "metrics": results_df.to_dict(orient="records"),
}
model_path = Path("../app/ml_model/world_cup_prediction_pipeline.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, model_path)
print("Modèle sauvegardé :", model_path)


## Modèles complémentaires pour le score

Deux régressions de Poisson estiment séparément les buts domicile et extérieur. Elles utilisent uniquement les mêmes informations disponibles avant le match. La MAE mesure l'écart moyen en buts ; le score affiché dans Django restera une estimation, pas une certitude.


In [ ]:
score_models = {}
score_metrics = {}
for side, target_column in [("home", "home_score"), ("away", "away_score")]:
    score_model = PoissonRegressor(alpha=0.01 if side == "home" else 0.10, max_iter=500)
    score_model.fit(X_train_scaled, data.loc[train_mask, target_column])
    test_goals = score_model.predict(X_test_scaled)
    score_metrics[f"{side}_goals_mae"] = mean_absolute_error(data.loc[test_mask, target_column], test_goals)
    score_model.fit(X_scaled_full, data[target_column])
    score_models[side] = score_model

score_bundle = {
    "home_model": score_models["home"],
    "away_model": score_models["away"],
    "feature_columns": feature_columns,
    "onehot_features": onehot_features,
    "encoded_columns": encoded_columns,
    "numeric_features": numeric_features,
    "scaler_mean": final_mean.to_dict(),
    "scaler_std": final_std.to_dict(),
    "metrics": score_metrics,
    "training_end_date": data["date"].max().strftime("%Y-%m-%d"),
}
score_model_path = Path("../app/ml_model/world_cup_score_pipeline.pkl")
joblib.dump(score_bundle, score_model_path)
print("MAE des modèles de buts :", score_metrics)
print("Modèle de score sauvegardé :", score_model_path)
